Kelvin Helmholtz instability

In [1]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from pygpe.shared.grid import Grid
from pygpe.scalar.wavefunction import ScalarWavefunction
from pygpe.scalar.evolution import step_wavefunction

In [2]:
pts = (256, 256)
dx, dy = (0.25, 0.25)
grid = Grid(pts, (dx, dy))

X = grid.x_mesh.get()
Y = grid.y_mesh.get()
Lx = pts[0] * dx
Ly = pts[1] * dy

n0 = 1.0
xi = 1.0
g = 1.0
dt = 0.005

In [3]:
A = 0.5               
lambda_pert = Lx / 3  

m = 4  
v0 = m * 2 * np.pi / Lx

Y1_center = -Ly / 4
Y2_center = Ly / 4

y1_bound = Y1_center + A * np.sin(2 * np.pi * X / lambda_pert)
y2_bound = Y2_center + A * np.sin(2 * np.pi * X / lambda_pert)

dens_profile = np.abs(np.tanh((Y - y1_bound) / xi) * np.tanh((Y - y2_bound) / xi))

phase = np.zeros_like(X)
center_mask = (Y > y1_bound) & (Y < y2_bound)

phase[center_mask] = v0 * X[center_mask]      
phase[~center_mask] = -v0 * X[~center_mask]   


psi_cpu = np.sqrt(n0) * dens_profile * np.exp(1j * phase)

wfn = ScalarWavefunction(grid)
wfn.component = cp.asarray(psi_cpu, dtype=cp.complex128)
wfn.fourier_component = cp.fft.fftn(wfn.component)

params = {'g': g, 'dt': dt, 'trap': cp.zeros(pts, dtype=cp.float64)}

In [4]:
n_frames = 300
steps_per_frame = 20

fig, (ax_dens, ax_phase) = plt.subplots(1, 2, figsize=(12, 5))

im_dens = ax_dens.imshow(np.abs(psi_cpu)**2, extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                         cmap='magma', origin='lower', vmin=0, vmax=1.1)
ax_dens.set_title(r"Density $|\psi|^2$")
plt.colorbar(im_dens, ax=ax_dens)

im_phase = ax_phase.imshow(np.angle(psi_cpu), extent=[-Lx/2, Lx/2, -Ly/2, Ly/2],
                           cmap='twilight', origin='lower', vmin=-np.pi, vmax=np.pi)
ax_phase.set_title(r"Phase")
plt.colorbar(im_phase, ax=ax_phase)

def update(frame):
    for _ in range(steps_per_frame):
        step_wavefunction(wfn, params)
    
    psi_current = wfn.component.get()
    
    im_dens.set_data(np.abs(psi_current)**2)
    im_phase.set_data(np.angle(psi_current))
    
    t = (frame + 1) * steps_per_frame * dt
    fig.suptitle(f"Time: {t:.2f}")
    
    return [im_dens, im_phase]

ani = FuncAnimation(fig, update, frames=n_frames, blit=True)
ani.save("kh_instability/kh_instability.html", writer='html')
plt.close()